# Session 15 — Capstone: Coupling Measures on a Kuramoto Dyad

**Author:** PPSP lab  **·**  **Date:** 2026-05-27  **·**  **Project:** Quantum Optimal Transport course  **·**  **Status:** 🔬 Teaching · open research

## Purpose

Assemble everything into a real, *unresolved* question:

> Can a quantum-information / quantum-OT coupling measure, applied to a synthetic
> oscillator dyad with known injected coupling, do at least as well as the standard
> classical baselines (phase-locking value, classical correlation)?

We do not promise a positive answer. The capstone is **honest**: in this synthetic
playground all measures track $K$ smoothly, but the *open* question is whether the
quantum machinery offers a real advantage — and if so, in which regime.

This session ties M2 (information geometry, S7), M3+M4 (transport / quantum coupling,
S13), and the dictionary thread back to a concrete neuroscience-flavoured benchmark.

## 0. What you'll be able to do

- Simulate a **synthetic Kuramoto dyad** with known injected coupling $K$ and stochastic noise.
- Map the oscillator phases to a **phase-coherent qubit state** $|\psi(t)\rangle = (|0\rangle + e^{i\theta(t)}|1\rangle)/\sqrt{2}$ and time-average to a bipartite density matrix $\rho_{AB}$.
- Compute four coupling measures: **quantum mutual information** (S7), **Bures-coupling** (S11), **PLV**, **classical correlation**.
- Sweep $K$ and watch all four rise — and read the limits.
- Reason honestly about the *direct-sum vs tensor-product trap*, the *baseline beating problem*, and the **open questions** that remain.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.quantum.composite import partial_trace, tensor as quantum_tensor
from qot_course.quantum_ot.capstone import (
    simulate_kuramoto_dyad, joint_density_matrix,
    plv, cosine_correlation, coupling_qmi, coupling_bures,
)

np.random.seed(42)
viz.use_course_style()

## 1. The synthetic Kuramoto dyad

Two stochastic oscillators with known natural frequencies $\omega_1, \omega_2$ and an
injected sinusoidal coupling of strength $K$:

$$
\mathrm{d}\theta_1 = (\omega_1 + K\sin(\theta_2 - \theta_1))\,\mathrm{d}t + \sigma\,\mathrm{d}W_1
$$
$$
\mathrm{d}\theta_2 = (\omega_2 + K\sin(\theta_1 - \theta_2))\,\mathrm{d}t + \sigma\,\mathrm{d}W_2
$$

For $K = 0$ the oscillators evolve independently; for large $K$ they phase-lock. The
noise $\sigma$ destroys perfect synchrony, which is good — it makes the experiment more
realistic.

We simulate three representative regimes: uncoupled ($K = 0$), weakly coupled ($K = 1$),
strongly coupled ($K = 4$).

In [ ]:
duration = 80.0
omega_1, omega_2 = 1.0, 1.2
K_values_demo = [0.0, 1.0, 4.0]
trajectories = {
    K: simulate_kuramoto_dyad(K=K, omega_1=omega_1, omega_2=omega_2,
                              duration=duration, seed=0)
    for K in K_values_demo
}

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, K in zip(axes, K_values_demo):
    t1, t2 = trajectories[K]
    time = np.arange(len(t1)) * 0.05
    ax.plot(time, np.sin(t1), color=viz.SOURCE_COLOR, lw=1.2, label=r"$\sin\theta_1$")
    ax.plot(time, np.sin(t2), color=viz.TARGET_COLOR, lw=1.2, label=r"$\sin\theta_2$", alpha=0.85)
    ax.set_ylabel(rf"$K = {K}$")
    ax.set_yticks([-1, 0, 1])
    if ax is axes[0]:
        ax.legend(loc="upper right", ncol=2)
axes[-1].set_xlabel("time")
fig.suptitle("Synthetic Kuramoto dyad: uncoupled $\\to$ weakly $\\to$ strongly coupled",
             fontsize=14, fontweight="bold", y=0.995)
plt.tight_layout()
plt.show()

**Read the figure.**

- **$K=0$**: the two oscillators drift at different frequencies — no visible
  synchronisation, phases continually slipping past each other.
- **$K=1$**: partial phase locking. The two traces are *similar* in shape but not
  identical; noise prevents perfect synchrony.
- **$K=4$**: strong locking. The two traces are essentially overlapping with a small
  phase offset.

This visual progression is what every coupling measure must reproduce numerically.

## 2. From phases to a bipartite density matrix

We map each oscillator's phase to a *phase-coherent qubit state*:

$$ |\psi(t)\rangle = \frac{1}{\sqrt 2}(|0\rangle + e^{i\theta(t)}|1\rangle). $$

This is a pure state on the equator of the Bloch sphere, at azimuth $\theta(t)$. The
joint state $|\Psi(t)\rangle = |\psi_A(t)\rangle \otimes |\psi_B(t)\rangle$ is a pure
product state in $\mathcal{H}_A \otimes \mathcal{H}_B$. Time-averaging the projector
gives a **bipartite density matrix**

$$ \rho_{AB} = \mathbb{E}_t[\,|\Psi(t)\rangle\langle\Psi(t)|\,]. $$

For uncoupled oscillators with uniformly drifting phases, $\rho_{AB} \to I_4/4$ (maximally
mixed). For phase-locked oscillators, the off-block coherence $\rho_{AB}[01, 10]$
becomes nonzero — exactly the same expectation $\langle e^{i(\theta_A - \theta_B)}\rangle$
that PLV computes. **PLV is a single matrix element of $\rho_{AB}$**; the quantum
measures use the *whole* matrix.

**A direct-sum vs tensor-product warning.** Building a "joint covariance" of the 4-vector
$(\cos\theta_A, \sin\theta_A, \cos\theta_B, \sin\theta_B)$ would give a 4-dim *block*
structure on a *direct-sum* space $\mathcal{H}_A \oplus \mathcal{H}_B$ — not what we
want. Our $|\psi_A\rangle \otimes |\psi_B\rangle$ construction sits in
$\mathcal{H}_A \otimes \mathcal{H}_B$, the genuine *tensor product*. The two
constructions look superficially similar (both 4×4) but encode the joint correlations
in fundamentally different ways.

In [ ]:
for K in K_values_demo:
    t1, t2 = trajectories[K]
    rho_ab = joint_density_matrix(t1, t2)
    rho_a = partial_trace(rho_ab, keep=[0], dims=[2, 2])
    rho_b = partial_trace(rho_ab, keep=[1], dims=[2, 2])
    print(f"K = {K}")
    print(f"  rho_AB diagonal = {np.diag(rho_ab).real.round(3).tolist()}")
    print(f"  off-block (01, 10) coherence = {rho_ab[1, 2]:.4f}")
    print(f"  marginal rho_A (Bloch x) = {(rho_a[0, 1] + rho_a[1, 0]).real:.3f}")
    print()

**Read the output.** At $K=0$, the off-block coherence and the marginal Bloch
component are essentially zero — both oscillators look maximally mixed to the
ensemble. As $K$ grows, the off-block (01, 10) coherence (the "phase-difference"
matrix element) and the marginals' coherences grow — *exactly the signature of
phase locking, encoded in the operator structure*.

## 3. Four coupling measures, side by side

| Name | Formula | Type |
|---|---|---|
| **Quantum MI** $I(A{:}B)$ | $S(\rho_{AB}) + S(\rho_B) + S(\rho_A) - $ ... wait it's $S(\rho_A) + S(\rho_B) - S(\rho_{AB})$ | information |
| **Bures-coupling** | $d_B(\rho_{AB}, \rho_A \otimes \rho_B)$ | transport |
| **PLV** | $\|\langle e^{i(\theta_A - \theta_B)}\rangle\|$ | classical, phase-only |
| **Cosine correlation** | $\mathrm{corr}(\cos\theta_A, \cos\theta_B)$ | classical, Euclidean-like |

All four are zero (or near zero) iff the two oscillators are *fully decoupled*. They
disagree on *how* they grow with coupling — that's the whole experiment.

In [ ]:
print(f"{'K':>5s}  {'QMI (nats)':>12s}  {'Bures':>10s}  {'PLV':>8s}  {'corr':>8s}")
print("-" * 50)
for K in K_values_demo:
    t1, t2 = trajectories[K]
    rho_ab = joint_density_matrix(t1, t2)
    print(f"{K:>5.1f}  "
          f"{coupling_qmi(rho_ab):>12.4f}  "
          f"{coupling_bures(rho_ab):>10.4f}  "
          f"{plv(t1, t2):>8.4f}  "
          f"{cosine_correlation(t1, t2):>8.4f}")

**Read the output.** All four measures rise with $K$, but at different rates and with
different sensitivities. PLV and cosine correlation are bounded in $[0, 1]$; QMI is
unbounded above (in principle) but here saturates around 1 bit because the marginals
are near $I/2$ (S7: max QMI bounded by $\log d_A + \log d_B - $ a marginal-entropy
term). Bures-coupling has its own range.

The *qualitative* agreement (all rise with $K$) is reassuring; the *quantitative*
differences are what would matter in a real application where ground truth is unknown.

## 4. Sweep $K$ — full picture

We run 25 simulations across $K \in [0, 4]$ and plot all four coupling measures.

In [ ]:
K_grid = np.linspace(0.0, 4.0, 25)
qmi_vals, bures_vals, plv_vals, corr_vals = [], [], [], []
for K in K_grid:
    t1, t2 = simulate_kuramoto_dyad(
        K=K, omega_1=omega_1, omega_2=omega_2, duration=200.0, seed=0,
    )
    rho_ab = joint_density_matrix(t1, t2)
    qmi_vals.append(coupling_qmi(rho_ab))
    bures_vals.append(coupling_bures(rho_ab))
    plv_vals.append(plv(t1, t2))
    corr_vals.append(abs(cosine_correlation(t1, t2)))

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
axes = axes.ravel()
panels = [
    ("Quantum mutual information $I(A{:}B)$  (nats)", qmi_vals, viz.SOURCE_COLOR),
    ("Bures-coupling  $d_B(\\rho_{AB}, \\rho_A \\otimes \\rho_B)$", bures_vals, viz.FLOW_COLOR),
    ("Phase-locking value  $\\mathrm{PLV}$", plv_vals, viz.TARGET_COLOR),
    ("Cosine correlation  $|\\mathrm{corr}(\\cos\\theta_A, \\cos\\theta_B)|$", corr_vals, "#0ea5e9"),
]
for ax, (title, vals, color) in zip(axes, panels):
    ax.plot(K_grid, vals, "-o", color=color, lw=2, markersize=4)
    ax.set_title(title, pad=8)
    ax.set_ylabel("coupling")
axes[2].set_xlabel("injected coupling  $K$")
axes[3].set_xlabel("injected coupling  $K$")
fig.suptitle("Coupling measures on a synthetic Kuramoto dyad — all four track $K$",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

**Read the four panels.**

- **Quantum mutual information**: starts near $0$ at $K = 0$ (independent oscillators ⇒ $\rho_{AB} \approx I/4$, marginals $\approx I/2$, so $I(A:B) \approx \log 2 + \log 2 - \log 4 = 0$). Climbs steeply with $K$ and saturates around $\log 2 \approx 0.69$ nats (= 1 bit) — the maximum for marginals constrained to $I/2$.
- **Bures-coupling**: rises monotonically with $K$, bounded above by $\sqrt 2$ (orthogonal states). A *transport-geometric* coupling measure, complementary to QMI.
- **PLV**: starts near $0$, rises to $\sim 0.9+$ at strong coupling. The classical neuroscience workhorse.
- **Cosine correlation**: similar shape to PLV but slightly noisier — sensitive to amplitude as well as phase.

**All four measures track the ground truth $K$.** The question of whether the quantum
measures *out-perform* the baselines requires careful experimental design — see the
honest caveats below.

## 5. Honest caveats — what this experiment does NOT prove

This synthetic study is a **sanity test**, not a validation of quantum coupling
measures as a clinical tool. Several issues stand between the picture above and an
actual claim of superiority over PLV / classical correlation:

1. **Estimation bias.** All measures here are computed from finite-length time series.
   QMI is particularly bias-prone (Treves & Panzeri, 1995): the estimated $S(\rho_{AB})$
   is systematically *lower* than the true value when $N$ is small. **Shrinkage**
   estimators or **panel-corrected** estimators are needed in practice.

2. **Leakage correction.** When the data come from filtered signals (EEG band-passing,
   wavelet decomposition), volume conduction and spectral leakage induce *spurious*
   couplings. The quantum machinery does not magic these away.

3. **Multiple comparisons / cherry-picking.** A single demonstration on a single
   synthetic example does not justify a general method. A serious test needs (a) a
   *family* of generative models (Kuramoto, Stuart–Landau, oscillator networks), (b)
   *replicated* simulations across noise realisations, (c) *baselines* with their own
   appropriate statistical machinery.

4. **The direct-sum vs tensor-product trap.** Our construction uses
   $|\psi_A\rangle \otimes |\psi_B\rangle$ — the *tensor product*. If we had built a
   joint covariance of $(\cos\theta_A, \sin\theta_A, \cos\theta_B, \sin\theta_B)$ instead
   (a *direct sum* in $\mathcal{H}_A \oplus \mathcal{H}_B$), we would get a different
   4×4 matrix and a different "coupling". Reasonable people make this mistake; the
   quantum literature has examples of both, with different theoretical guarantees.

5. **Phase only.** Our $|\psi(t)\rangle$ throws away the *amplitude* of the oscillator.
   For purely-phase coupling like Kuramoto this is fine; for amplitude-coupled systems
   (e.g. cross-frequency phase-amplitude coupling), the construction would need to be
   generalised.

These are the **open questions** that make this an active research area. The course
ends honestly: we have built a beautiful set of tools, and a careful, conservative
demonstration of one of their possible applications. The next step is *experimental
methodology*, not more mathematics.

## 6. The completed picture — what the course built

Across 15 sessions we have constructed:

- A **classical ↔ quantum dictionary** (S1 → S12) covering every major information- and transport-theoretic object.
- The **Wasserstein / Bures bridge** (S11) explicitly linking covariance matrices and density matrices.
- The **Sinkhorn / quantum Sinkhorn pair** (S10, S14) showing that the Bregman geometry on the PSD cone is structurally identical to the orthant case.
- The **quantum-OT SDP** (S13) and its entropic regularisation (S14) as the principal computational definition.
- An **honest open research problem** (S15) tying the machinery to oscillator coupling, with all caveats explicit.

The next session (S16) closes the loop with the **frontier and a synthesis**: the
naive-VQE limitation result (De Palma et al., 2023, on a class of *quantum* OT
formulations that admit polynomial-time *classical* algorithms — closing the loop with
the original broken demo of the course's prologue), the qubit-$W_1$ formulation
(De Palma, Marvian, Trevisan, Lloyd, 2021), and remaining open problems.

## 7. Your turn

1. **Replication across noise.** Re-run the K sweep with 10 different `seed` values and
   plot the mean ± std of each measure. How much does the noise wash out the curve?
   *Which measure is most/least noise-robust?*
2. **Direct-sum vs tensor-product.** Build the alternative 4×4 matrix from the
   covariance of $(\cos\theta_A, \sin\theta_A, \cos\theta_B, \sin\theta_B)$. Compute
   QMI on it (treating it as a "density matrix" after normalisation) and compare to our
   tensor-product version. Do they tell the same story?
3. **What would beat PLV?** Construct a coupled system where two oscillators have the
   same phase-difference *distribution* (so PLV is identical) but different
   higher-order joint structure (e.g. different amplitude coupling). Show that the
   quantum measures distinguish them.

## Conclusions & references

- The synthetic Kuramoto dyad provides ground truth ($K$) against which all coupling
  measures can be calibrated.
- Our **phase-coherent qubit construction** gives a bipartite density matrix that
  cleanly encodes phase locking as the off-block coherence $\rho_{AB}[01, 10]$.
- All four measures — **quantum MI**, **Bures-coupling**, **PLV**, **classical
  correlation** — track $K$ monotonically. None is obviously superior in this regime.
- The **direct-sum vs tensor-product trap** is real: superficially-similar 4×4 matrices
  can encode very different physics.
- **The open question.** Does the quantum machinery provide a *real* advantage over
  PLV / classical correlation? In which regime? What is the statistical price (sample
  size, estimator bias) for that advantage? *We do not know.*
- **Next (S16 — Frontier & synthesis):** the De Palma et al. (2023) VQE limitation
  result, qubit-$W_1$ (De Palma, Marvian, Trevisan, Lloyd 2021), open problems
  (quantum TE, hyperscanning QOT), and the final course synthesis.

**References:** Y. Kuramoto, *Chemical Oscillations, Waves, and Turbulence* (Springer,
1984); J.-P. Lachaux, E. Rodriguez, J. Martinerie, F. Varela, "Measuring phase synchrony
in brain signals", *Hum. Brain Mapp.* **8**, 194 (1999); M. Treves, S. Panzeri, "The
upward bias in measures of information derived from limited data samples",
*Neural Comput.* **7**, 399 (1995); D. Trevisan, *Optimal transport methods for quantum
systems* (arXiv:2202.02091, 2022); De Palma & Trevisan, *Ann. Henri Poincaré*
**22**, 3199 (2021). Previous: `notebooks/s14_quantum_sinkhorn.ipynb`.